# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jayanthGowda1718/ml-internship-work/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is engagement_fix. This is a classification task: predicting whether a page has a fixable engagement/CTR problem or not.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Proxy target: needs_engagement_fix = 1 if a page's CTR is well below the expected CTR for its rank position, calculated from impressions, clicks, and average position in the data. There's no direct "would fixing this help" label, so this proxy is built from existing columns.

In [6]:

expected_ctr = df.groupby("position_tier")["ctr"].transform("median")
df["needs_engagement_fix"] = (df["ctr"] < 0.6 * expected_ctr).astype(int)

df["needs_engagement_fix"].value_counts()

,count
needs_engagement_fix,
0,18417
1,11583


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50 — of the top 50 pages flagged as needing an engagement fix, how many actually have a genuine, fixable CTR gap. This matters because a content team can only manually review a limited number of pages per week, so precision at the top of the queue is what determines whether their time is well spent.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page (URL-level, anonymized). Each row has that page's search performance metrics (impressions, clicks, CTR, average rank position) which we use to flag pages with an engagement gap.

In [8]:
import pandas as pd

url = "https://raw.githubusercontent.com/jayanthGowda1718/ml-internship-work/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [9]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [10]:
df[["content_id", "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier"]].head(10)

,content_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier
0,content_304f48230142,3803,29,0.76,10.6,striking
1,content_a1fb4e703a9e,15320,7,0.05,20.3,page_3_5
2,content_9aa793d4d895,12581,11,0.09,36.5,page_3_5
3,content_331d6c4de07b,11751,58,0.49,6.2,page_1
4,content_d99b7a2d90ca,19140,24,0.13,44.0,page_3_5
5,content_d4084a4bc775,3970,1,0.03,8.5,page_1
6,content_9a34b442b552,20,0,0.00,7.0,page_1
7,content_a63219c6e95a,1724,1,0.06,21.2,page_3_5
8,content_5e6c160719bc,32574,29,0.09,46.0,page_3_5
9,content_c27558df2b0c,1240,2,0.16,4.9,page_1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag any page with CTR below 2%" ignores that expected CTR varies hugely by rank position — position 1 naturally gets ~30% CTR, position 10 gets only ~2%. A flat threshold either flags almost every low-ranked page as broken (too much noise) or misses high-ranked pages that genuinely have a CTR problem. The relationship between position and expected CTR isn't a clean cutoff — it's a curve that changes shape depending on query type, device, and content category. ML can learn this expected curve directly from the data and flag real deviations from it, which a single if-statement threshold can't capture without secretly hardcoding that same curve by hand.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.